## In Class Activity April 21st

### Libraries, data import & EDA

In [ ]:
# importing libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score
import sweetviz as sv
from sklearn.model_selection import cross_val_score
from sklearn.compose import make_column_selector, ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import lightgbm as lgbm
import optuna

In [98]:
# importing the data
adult = pd.read_csv("adult.csv")
adult.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [61]:
# quick eda with sweetviz
report = sv.analyze(adult)
report.show_html("sweetviz_report.html")

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report sweetviz_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### Model agnostic data preprocessing & train/test split for reuse in all cells

In [99]:
# replace ? with np.nan
adult = adult.replace("?", np.nan)
adult.head()

# convert target variable to binary
adult["income"] = adult["income"].apply(lambda x: 1 if x == ">50K" else 0)

# convert gender to 0/1 (doesn't need categorical encoding since it's binary)
if "gender" in adult.columns:
    adult["gender"] = adult["gender"].apply(lambda x: 1 if x == "Male" else 0)  
    
adult.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,0,0,0,30,United-States,0


In [100]:
#define X and y
cat_columns = ['workclass',
 'education',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'native-country']
adult[cat_columns] = adult[cat_columns].astype("category")
X = adult.drop(columns = ["income"])
y = adult["income"]

In [101]:
X.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,0,0,0,30,United-States


In [102]:
y.head()

0    0
1    0
2    1
3    1
4    0
Name: income, dtype: int64

In [103]:
#define train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 321, stratify = y)

## Baseline modeling with CV

In [76]:
X.dtypes

age                   int64
workclass          category
fnlwgt                int64
education          category
educational-num       int64
marital-status     category
occupation         category
relationship       category
race               category
gender                int64
capital-gain          int64
capital-loss          int64
hours-per-week        int64
native-country     category
dtype: object

In [104]:
#dummify categorical variables (for models that cannot handle categoricals natively)
ct_dummify = ColumnTransformer(
  [
    ("dummify",
    OneHotEncoder(sparse_output = False, handle_unknown='ignore'),
    make_column_selector(dtype_include=["category"]))],
  remainder = "passthrough"
).set_output(transform="pandas")

ct_dummify.fit_transform(X).head()

,dummify__workclass_Federal-gov,dummify__workclass_Local-gov,dummify__workclass_Never-worked,dummify__workclass_Private,dummify__workclass_Self-emp-inc,dummify__workclass_Self-emp-not-inc,dummify__workclass_State-gov,dummify__workclass_Without-pay,dummify__workclass_nan,dummify__education_10th,...,dummify__native-country_Vietnam,dummify__native-country_Yugoslavia,dummify__native-country_nan,remainder__age,remainder__fnlwgt,remainder__educational-num,remainder__gender,remainder__capital-gain,remainder__capital-loss,remainder__hours-per-week
0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,25,226802,7,1,0,0,40
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,38,89814,9,1,0,0,50
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,28,336951,12,1,0,0,40
3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,44,160323,10,1,7688,0,40
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,18,103497,10,0,0,0,30


In [105]:
# baseline models with CV
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=321)

# Random Forest
rf_pipeline = Pipeline([
    ("preprocessing", ct_dummify),
    ("rf", RandomForestClassifier(random_state=321, class_weight='balanced', n_jobs=-1))
])

rf_cv_scores = cross_val_score(rf_pipeline, X_train, y_train, cv=kf, scoring="roc_auc")
print(f"RF CV ROC AUC: {rf_cv_scores.mean():.4f} ± {rf_cv_scores.std():.4f}")

rf_pipeline.fit(X_train, y_train)
rf_proba = rf_pipeline.predict_proba(X_test)[:, 1]
print(f"RF Test ROC AUC: {roc_auc_score(y_test, rf_proba):.4f}")
print("RF Classification Report:\n", classification_report(y_test, rf_pipeline.predict(X_test)))

# LightGBM
classes, counts = np.unique(y, return_counts=True)
class_weights = dict(zip(classes.tolist(), (len(y) / (len(classes) * counts)).tolist()))

lgb_pipeline = Pipeline([
    ("preprocessing", ct_dummify),
    ("lgbm", lgbm.LGBMClassifier(random_state=321, class_weight=class_weights, verbose=0))
])

lgb_cv_scores = cross_val_score(lgb_pipeline, X_train, y_train, cv=kf, scoring="roc_auc")
print(f"lgbm CV ROC AUC: {lgb_cv_scores.mean():.4f} ± {lgb_cv_scores.std():.4f}")

lgb_pipeline.fit(X_train, y_train)
lgb_proba = lgb_pipeline.predict_proba(X_test)[:, 1]
print(f"lgbm Test ROC AUC: {roc_auc_score(y_test, lgb_proba):.4f}")
print("lgbm Classification Report:\n", classification_report(y_test, lgb_pipeline.predict(X_test)))

RF CV ROC AUC: 0.9026 ± 0.0025
RF Test ROC AUC: 0.9018
RF Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.93      0.91      7431
           1       0.73      0.60      0.66      2338

    accuracy                           0.85      9769
   macro avg       0.81      0.77      0.78      9769
weighted avg       0.85      0.85      0.85      9769

lgbm CV ROC AUC: 0.9292 ± 0.0020
lgbm Test ROC AUC: 0.9287
lgbm Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.82      0.88      7431
           1       0.61      0.86      0.71      2338

    accuracy                           0.83      9769
   macro avg       0.78      0.84      0.80      9769
weighted avg       0.87      0.83      0.84      9769



### Which features are important?

In [106]:
#extract post-encoding feature names from the pipeline
feature_names = [
    name.split('__')[-1]
    for name in rf_pipeline.named_steps['preprocessing'].get_feature_names_out()
]

#extract importances 
rf_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_pipeline.named_steps['rf'].feature_importances_
}).sort_values(by="importance", ascending=False).reset_index(drop=True)

lgb_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": lgb_pipeline.named_steps['lgbm'].feature_importances_
}).sort_values(by="importance", ascending=False).reset_index(drop=True)

print("Top 10 RF Features:\n",  rf_importance_df.head(10))
print("\nTop 10 lgbm Features:\n", lgb_importance_df.head(10))

Top 10 RF Features:
                              feature  importance
0                                age    0.151552
1                             fnlwgt    0.142316
2  marital-status_Married-civ-spouse    0.089446
3                     hours-per-week    0.084236
4                       capital-gain    0.067335
5                    educational-num    0.061058
6               relationship_Husband    0.053521
7       marital-status_Never-married    0.037857
8                       capital-loss    0.020425
9             relationship_Own-child    0.017619

Top 10 lgbm Features:
                              feature  importance
0                                age         477
1                             fnlwgt         352
2                       capital-loss         295
3                       capital-gain         287
4                     hours-per-week         265
5                    educational-num         208
6  marital-status_Married-civ-spouse          66
7         occupation_Exe

## Feature engineering

In [107]:
#net capital activity - like observing an interaction between the two
adult['capital_net'] = adult['capital-gain'] - adult['capital-loss'] 

#age and education interaction
adult['age_education_interaction'] = adult['age'] * adult['educational-num']

#log transformation of net capital activity to help models handle extreme values
capital_net_clipped = adult['capital_net'].clip(lower=0)
adult['capital_net_log'] = np.log1p(capital_net_clipped)

#age and marriage interaction
adult['age_married'] = adult['age'] * (adult['marital-status'] == 'Married-civ-spouse').astype(int)

## Re-run baseline models with new features

In [108]:
#redefine X and y
X = adult.drop(columns=['income'])
y = adult['income']

In [82]:
X.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,capital_net,age_education_interaction,capital_net_log,age_married
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0,175,0.000000,0
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0,342,0.000000,38
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,0,336,0.000000,28
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,7688,440,8.947546,44
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,0,0,0,30,United-States,0,180,0.000000,0


In [83]:
y.head()

0    0
1    0
2    1
3    1
4    0
Name: income, dtype: int64

In [109]:
#redefine train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=321, stratify=y
)

In [85]:
X.dtypes

age                             int64
workclass                    category
fnlwgt                          int64
education                    category
educational-num                 int64
marital-status               category
occupation                   category
relationship                 category
race                         category
gender                          int64
capital-gain                    int64
capital-loss                    int64
hours-per-week                  int64
native-country               category
capital_net                     int64
age_education_interaction       int64
capital_net_log               float64
age_married                     int64
dtype: object

In [110]:
# baseline models with CV
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=321)

# Random Forest
rf_pipeline = Pipeline([
    ("preprocessing", ct_dummify),
    ("rf", RandomForestClassifier(random_state=321, class_weight='balanced', n_jobs=-1))
])

rf_cv_scores = cross_val_score(rf_pipeline, X_train, y_train, cv=kf, scoring="roc_auc")
print(f"RF CV ROC AUC: {rf_cv_scores.mean():.4f} ± {rf_cv_scores.std():.4f}")

rf_pipeline.fit(X_train, y_train)
rf_proba = rf_pipeline.predict_proba(X_test)[:, 1]
print(f"RF Test ROC AUC: {roc_auc_score(y_test, rf_proba):.4f}")
print("RF Classification Report:\n", classification_report(y_test, rf_pipeline.predict(X_test)))

# LightGBM
classes, counts = np.unique(y, return_counts=True)
class_weights = dict(zip(classes.tolist(), (len(y) / (len(classes) * counts)).tolist()))

lgb_pipeline = Pipeline([
    ("preprocessing", ct_dummify),
    ("lgbm", lgbm.LGBMClassifier(random_state=321, class_weight=class_weights, verbose=0))
])

lgb_cv_scores = cross_val_score(lgb_pipeline, X_train, y_train, cv=kf, scoring="roc_auc")
print(f"lgbm CV ROC AUC: {lgb_cv_scores.mean():.4f} ± {lgb_cv_scores.std():.4f}")

lgb_pipeline.fit(X_train, y_train)
lgb_proba = lgb_pipeline.predict_proba(X_test)[:, 1]
print(f"lgbm Test ROC AUC: {roc_auc_score(y_test, lgb_proba):.4f}")
print("lgbm Classification Report:\n", classification_report(y_test, lgb_pipeline.predict(X_test)))

RF CV ROC AUC: 0.9055 ± 0.0019
RF Test ROC AUC: 0.9041
RF Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.93      0.91      7431
           1       0.73      0.61      0.66      2338

    accuracy                           0.85      9769
   macro avg       0.81      0.77      0.79      9769
weighted avg       0.85      0.85      0.85      9769

lgbm CV ROC AUC: 0.9289 ± 0.0021
lgbm Test ROC AUC: 0.9282
lgbm Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.82      0.88      7431
           1       0.60      0.87      0.71      2338

    accuracy                           0.83      9769
   macro avg       0.78      0.84      0.80      9769
weighted avg       0.87      0.83      0.84      9769



## Check which features are important again

In [111]:
#extract post-encoding feature names from the pipeline
feature_names = [
    name.split('__')[-1]
    for name in rf_pipeline.named_steps['preprocessing'].get_feature_names_out()
]

#extract importances 
rf_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_pipeline.named_steps['rf'].feature_importances_
}).sort_values(by="importance", ascending=False).reset_index(drop=True)

lgb_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": lgb_pipeline.named_steps['lgbm'].feature_importances_
}).sort_values(by="importance", ascending=False).reset_index(drop=True)

print("Top 10 RF Features:\n",  rf_importance_df.head(10))
print("\nTop 10 lgbm Features:\n", lgb_importance_df.head(10))

Top 10 RF Features:
                              feature  importance
0                             fnlwgt    0.111434
1                        age_married    0.110327
2          age_education_interaction    0.094004
3                                age    0.081443
4                     hours-per-week    0.068643
5               relationship_Husband    0.056365
6                    educational-num    0.045182
7  marital-status_Married-civ-spouse    0.044034
8                       capital-gain    0.034338
9                        capital_net    0.034091

Top 10 lgbm Features:
                       feature  importance
0   age_education_interaction         341
1                      fnlwgt         337
2                capital-gain         325
3              hours-per-week         239
4                capital-loss         233
5                         age         232
6                 age_married         157
7             educational-num         110
8                 capital_net         

## Model tuning

In [112]:
#random forest tuning
def rf_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300, step=100),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
    }

    rf = Pipeline([
        ("preprocessing", ct_dummify),
        ("rf", RandomForestClassifier(
            **params,
            random_state=321,
            class_weight='balanced',
            n_jobs=-1
        ))
    ])

    score = cross_val_score(
        rf, X_train, y_train,
        cv=kf,
        scoring='roc_auc',
        n_jobs=-1
    ).mean()

    return score

rf_study = optuna.create_study(direction='maximize')
rf_study.optimize(rf_objective, n_trials=20, show_progress_bar=True)

print("Best RF AUC:    ", round(rf_study.best_value, 4))
print("Best RF Params: ", rf_study.best_params)

# refit with best params and evaluate on test set
best_rf_pipeline = Pipeline([
    ("preprocessing", ct_dummify),
    ("rf", RandomForestClassifier(
        **rf_study.best_params,
        random_state=321,
        class_weight='balanced',
        n_jobs=-1
    ))
])

best_rf_pipeline.fit(X_train, y_train)
rf_tuned_proba = best_rf_pipeline.predict_proba(X_test)[:, 1]
print(f"Tuned RF Test AUC: {roc_auc_score(y_test, rf_tuned_proba):.4f}")

[I 2026-04-22 18:39:20,490] A new study created in memory with name: no-name-44052814-841b-4ef3-855a-ac6f15119d08
Best trial: 0. Best value: 0.913708:   5%|▌         | 1/20 [00:07<02:25,  7.64s/it]

[I 2026-04-22 18:39:28,128] Trial 0 finished with value: 0.9137078546805982 and parameters: {'n_estimators': 100, 'max_depth': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.9137078546805982.


Best trial: 0. Best value: 0.913708:  10%|█         | 2/20 [00:13<01:54,  6.35s/it]

[I 2026-04-22 18:39:33,583] Trial 1 finished with value: 0.9135122305102442 and parameters: {'n_estimators': 100, 'max_depth': 13, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 0 with value: 0.9137078546805982.


Best trial: 2. Best value: 0.916675:  15%|█▌        | 3/20 [00:16<01:27,  5.13s/it]

...

[I 2026-04-22 18:41:18,712] Trial 17 finished with value: 0.9174946303019956 and parameters: {'n_estimators': 200, 'max_depth': 18, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 15 with value: 0.9178772834104741.


Best trial: 15. Best value: 0.917877:  95%|█████████▌| 19/20 [02:07<00:07,  7.26s/it]

[I 2026-04-22 18:41:27,535] Trial 18 finished with value: 0.9161751567633362 and parameters: {'n_estimators': 300, 'max_depth': 15, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 15 with value: 0.9178772834104741.


Best trial: 15. Best value: 0.917877: 100%|██████████| 20/20 [02:13<00:00,  6.68s/it]


[I 2026-04-22 18:41:34,031] Trial 19 finished with value: 0.9170314848605405 and parameters: {'n_estimators': 200, 'max_depth': 20, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 15 with value: 0.9178772834104741.
Best RF AUC:     0.9179
Best RF Params:  {'n_estimators': 200, 'max_depth': 20, 'min_samples_leaf': 2, 'max_features': 'log2'}
Tuned RF Test AUC: 0.9169


In [115]:
def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'num_leaves': trial.suggest_int('num_leaves', 20, 60),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
    }

    lgb_pipe = Pipeline([
        ("preprocessing", ct_dummify),
        ("lgbm", lgbm.LGBMClassifier(
            **params,
            random_state=321,
            class_weight=class_weights,
            verbose=0,
            n_jobs=-1
        ))
    ])

    score = cross_val_score(
        lgb_pipe, X_train, y_train,
        cv=kf,
        scoring='roc_auc',
        n_jobs=-1
    ).mean()

    return score

lgb_study = optuna.create_study(direction='maximize')
lgb_study.optimize(lgb_objective, n_trials=20, show_progress_bar=True)

print("Best lgbm AUC:    ", round(lgb_study.best_value, 4))
print("Best lgbm Params: ", lgb_study.best_params)

best_lgb_pipeline = Pipeline([
    ("preprocessing", ct_dummify),
    ("lgbm", lgbm.LGBMClassifier(
        **lgb_study.best_params,
        random_state=321,
        class_weight=class_weights,
        verbose=0,
        n_jobs=-1
    ))
])

best_lgb_pipeline.fit(X_train, y_train)
lgb_tuned_proba = best_lgb_pipeline.predict_proba(X_test)[:, 1]
print(f"Tuned lgbm Test AUC: {roc_auc_score(y_test, lgb_tuned_proba):.4f}")

[I 2026-04-22 18:43:05,587] A new study created in memory with name: no-name-e751347a-d2c9-4c09-9a6b-34b6d738b60a
Best trial: 0. Best value: 0.922797:   5%|▌         | 1/20 [00:02<00:46,  2.42s/it]

[I 2026-04-22 18:43:08,009] Trial 0 finished with value: 0.922796930648202 and parameters: {'n_estimators': 200, 'learning_rate': 0.03057432797414571, 'max_depth': 5, 'num_leaves': 55, 'min_child_samples': 50}. Best is trial 0 with value: 0.922796930648202.


Best trial: 1. Best value: 0.928527:  10%|█         | 2/20 [00:04<00:40,  2.27s/it]

[I 2026-04-22 18:43:10,179] Trial 1 finished with value: 0.9285273131034243 and parameters: {'n_estimators': 100, 'learning_rate': 0.06852852820638347, 'max_depth': 14, 'num_leaves': 56, 'min_child_samples': 32}. Best is trial 1 with value: 0.9285273131034243.


Best trial: 2. Best value: 0.928569:  15%|█▌        | 3/20 [00:09<00:55,  3.29s/it]

...

[I 2026-04-22 18:43:48,512] Trial 17 finished with value: 0.9281614227451144 and parameters: {'n_estimators': 200, 'learning_rate': 0.0396129651167581, 'max_depth': 7, 'num_leaves': 41, 'min_child_samples': 17}. Best is trial 13 with value: 0.9291970532247816.


Best trial: 13. Best value: 0.929197:  95%|█████████▌| 19/20 [00:45<00:02,  2.37s/it]

[I 2026-04-22 18:43:50,673] Trial 18 finished with value: 0.9286220745809185 and parameters: {'n_estimators': 100, 'learning_rate': 0.06965867920390327, 'max_depth': 12, 'num_leaves': 39, 'min_child_samples': 23}. Best is trial 13 with value: 0.9291970532247816.


Best trial: 13. Best value: 0.929197: 100%|██████████| 20/20 [00:48<00:00,  2.45s/it]


[I 2026-04-22 18:43:54,509] Trial 19 finished with value: 0.9273370807474789 and parameters: {'n_estimators': 300, 'learning_rate': 0.06971225314248333, 'max_depth': 16, 'num_leaves': 45, 'min_child_samples': 15}. Best is trial 13 with value: 0.9291970532247816.
Best lgbm AUC:     0.9292
Best lgbm Params:  {'n_estimators': 200, 'learning_rate': 0.0657507869775994, 'max_depth': 9, 'num_leaves': 20, 'min_child_samples': 10}
Tuned lgbm Test AUC: 0.9289


### Stacking predictions-average

In [ ]:
# building ensemble of RF and LGB

#averages predicted probabilities from both models for each observation (incorporates both models' opinions with equal weight)
avg_probs = (rf_tuned_proba + lgb_tuned_proba) / 2

#converts probabilities into hard class predictions
y_pred_ensemble = (avg_probs >= 0.5).astype(int)

print(f"Baseline RF AUC:  {roc_auc_score(y_test, rf_proba):.4f}")
print(f"Baseline LGB AUC: {roc_auc_score(y_test, lgb_proba):.4f}")
print(f"Tuned RF AUC:     {roc_auc_score(y_test, rf_tuned_proba):.4f}")
print(f"Tuned LGB AUC:    {roc_auc_score(y_test, lgb_tuned_proba):.4f}")
print(f"Ensemble AUC:     {roc_auc_score(y_test, avg_probs):.4f}")

print("\nEnsemble Classification Report:")
print(classification_report(y_test, y_pred_ensemble))

Baseline RF AUC:  0.9041
Baseline LGB AUC: 0.9282
Tuned RF AUC:     0.9169
Tuned LGB AUC:    0.9289
Ensemble AUC:     0.9256

Ensemble Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.83      0.88      7431
           1       0.61      0.86      0.71      2338

    accuracy                           0.83      9769
   macro avg       0.78      0.84      0.80      9769
weighted avg       0.87      0.83      0.84      9769



## Final evaluation

**Feature importance**
- Random forest: It's interesting to see that the age interactions (age * married and age * education) are identified as more important in the feature performance of this model. It seems like age is lower in feature performance after taking into account these new interaction terms. It also seems like the importance of capital gain went down after creating the net capital feature. Other features that remained important include fnlwgt, hours worked per week, and education (numeric)

- LGB: Similar to the random forest model, the age interaction with education gained more importance compared to the age feature itself. It appears that the other engineered features did not change much in terms of feature importance. Other important features for this model include fnlwgt, capital gain, and hours worked per week.

Overall, the random forest model performed better with the engineered features compared to the baseline model (0.904 vs 0.902).

**Best model**
- The best model was the LGB model with 200 estimators, a learning rate of 0.066, max depth of 9, 20 leaves, and 10 minimum observations in a leaf (ROC AUC score of 0.929). These hyperparameter values are quite moderate, indicating that the data does not require highly complex models to generalize well.

**Ensemble performance**

The ensemble model with equal influence of the random forest and LGB models performed quite well (ROC AUC score of 0.926), but not as well as the LGB models. This is likely due to the inclusion of the random forest model. The ensemble model could potentially perform better with more weight on the LGB model.

**Takeaway**

It was really helpful to see the entire process of baseline modeling, to feature engineering, to tuning, and ensemble modeling. This is definitely a process I will follow in my future assignments and work. My only concern is that the feature engineering did not seem to substantially improve my models and it took a lot of effort. I think I will only spend time on feature engineering if the other methods of model improvement (i.e. tuning, ensembles) does not result in much improvement.